In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

import sys
import json
# -----------------------------
# Ajouter le repo au Python Path
# -----------------------------
sys.path.append("/Workspace/Users/mandu543@gmail.com/databricks_flights")

from src.Communs.utils import *
from conf.config import *

## Créer une alimentation Dynamique des tables de Fait depuis Silver vers Gold Layer

### Paramètres

In [0]:
# ============================================================
# 1. RÉCUPÉRATION DES PARAMÈTRES
# ============================================================
source_schema = dbutils.widgets.get("source_schema")
source_table  = dbutils.widgets.get("source_table")
target_schema = dbutils.widgets.get("target_schema")
target_table  = dbutils.widgets.get("target_table")
fact_columns  = dbutils.widgets.get("fact_columns").split(",")   # string -> liste
dimensions    = json.loads(dbutils.widgets.get("dimensions"))    # string -> dict
cdc_col       = dbutils.widgets.get("cdc_col")
surrogate_col = dbutils.widgets.get("surrogate_col")

print(f"Parametres : {source_schema}, {source_table}, {target_schema}, {target_table}, {fact_columns}, {dimensions}, {cdc_col}, {surrogate_col}")


### Création de la fonction

In [0]:

# ============================================================
# 2. FONCTION
# ============================================================
def incremental_fact_load(
   source_schema, source_table,
   target_schema, target_table,
   fact_columns, dimensions,
   cdc_col, surrogate_col,
   backdated_refresh=None
):
   source_full = f"{source_schema}.{source_table}"
   target_full = f"{target_schema}.{target_table}"
   fact_id_sk     = f"{target_table}_fact_id"
   # ----------------------------------------------------------
   # LAST LOAD
   # ----------------------------------------------------------
   if backdated_refresh:
       last_load = backdated_refresh
   elif spark.catalog.tableExists(target_full):
       last_load = spark.table(target_full).select(max(cdc_col)).first()[0] or "1900-01-01"
   else:
       last_load = "1900-01-01"
   print(f"--- Chargement des données depuis : {last_load} ---")
   # ----------------------------------------------------------
   # LECTURE SOURCE — fact_columns + clés de jointure + cdc
   # ----------------------------------------------------------
   join_keys_needed = [dim["key"] for dim in dimensions]
   cols_to_select   = list(set(fact_columns + join_keys_needed + [cdc_col]))
   print(f"Colonnes à sélectionner : {cols_to_select}")
   


   df_final = spark.table(source_full) \
                   .filter(col(cdc_col) >= last_load) \
                   .select(*cols_to_select)
   if df_final.count() == 0:
       print("Aucune donnee a traiter.")
       return
   
   display(df_final)
   # ----------------------------------------------------------
   # ENRICHISSEMENT + SUPPRESSION DES CLÉS NATURELLES
   # On garde uniquement les surrogate keys en Gold (bonne pratique Kimball)
   # ----------------------------------------------------------
   for dim in dimensions:
       print(f"Jointure avec : {dim['table']}")
       df_dim = spark.table(dim["table"]) \
                     .select(col(dim["key"]).alias("dim_key"),
                             col(surrogate_col).alias(f"{dim['alias']}_{surrogate_col}"))
       df_final = df_final.join(df_dim, df_final[dim["key"]] == df_dim["dim_key"], "left") \
                          .drop("dim_key") \
                          .drop(dim["key"])  # clé naturelle supprimée
   display(df_final)
   # ----------------------------------------------------------
   # SURROGATE KEY FACT
   # ----------------------------------------------------------
   max_id = spark.table(target_full).select(max(fact_id_sk)).first()[0] or 0 \
            if spark.catalog.tableExists(target_full) else 0
   df_final = df_final \
       .withColumn(fact_id_sk, row_number().over(Window.orderBy(cdc_col)) + max_id) \
       .withColumn("load_timestamp", current_timestamp())
  
   # Ordonnancement des colonnes : fact_id_sk | métriques | surrogate keys dims | cdc | timestamp
   other_cols = [c for c in df_final.columns if c not in [fact_id_sk] + fact_columns]
   df_final   = df_final.select(fact_id_sk, *fact_columns, *other_cols)
   
   display(df_final)
   # ----------------------------------------------------------
   # MERGE / WRITE
   # ----------------------------------------------------------
   if spark.catalog.tableExists(target_full):
       print("Mise a jour de la table Gold existante (MERGE)...")
       DeltaTable.forName(spark, target_full).alias("t") \
           .merge(df_final.alias("s"), f"t.{fact_id_sk} = s.{fact_id_sk}") \
           .whenMatchedUpdateAll(condition=f"s.{cdc_col} >= t.{cdc_col}") \
           .whenNotMatchedInsertAll() \
           .execute()
   else:
       print("Creation de la table Gold (Initial Load)...")
       df_final.write.format("delta").mode("overwrite").saveAsTable(target_full)
   print("--- Fin du processus avec succes ---")
 

### Appel de la fonction

In [0]:
# ============================================================
# 2. APPEL DE LA FONCTION
# ============================================================
incremental_fact_load(
   source_schema = source_schema,
   source_table  = source_table,
   target_schema = target_schema,
   target_table  = target_table,
   fact_columns  = fact_columns,
   dimensions    = dimensions,
   cdc_col       = cdc_col,
   surrogate_col = surrogate_col
)